In [5]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.results_plotter import load_results, ts2xy
import matplotlib.pyplot as plt
import pandas as pd
from collections import deque
import os
import random

class MahimahiTraceManager:
    """
    Mengelola banyak file trace Mahimahi dari sebuah folder (.log atau .txt).
    Mengonversi timestamp milidetik Mahimahi menjadi array Throughput (Mbps) per detik.
    """
    def __init__(self, folder_path="traces_folder"):
        self.traces = []
        PACKET_SIZE_BITS = 1500 * 8 # Asumsi standar MTU 1500 bytes (Ethernet)
        
        if os.path.exists(folder_path):
            files = [f for f in os.listdir(folder_path) if f.endswith('.txt') or f.endswith('.log')]
            for file in files:
                path = os.path.join(folder_path, file)
                try:
                    with open(path, 'r') as f:
                        # Baca semua baris, abaikan yang kosong
                        timestamps_ms = [float(line.strip()) for line in f if line.strip()]
                        
                        if timestamps_ms:
                            # --- KONVERSI MAHIMAHI (Timestamp ms) -> Mbps ---
                            # Algoritma: Menghitung jumlah paket yang lewat dalam jendela 1 detik
                            throughput_mbps = []
                            current_sec = 0
                            packet_count = 0
                            
                            for ts in timestamps_ms:
                                sec = int(ts / 1000)
                                while current_sec < sec:
                                    mbps = (packet_count * PACKET_SIZE_BITS) / 1_000_000
                                    throughput_mbps.append(mbps)
                                    packet_count = 0
                                    current_sec += 1
                                packet_count += 1
                                
                            throughput_mbps.append((packet_count * PACKET_SIZE_BITS) / 1_000_000)
                            throughput_mbps = [max(0.1, tp) for tp in throughput_mbps]
                            
                            self.traces.append({
                                "name": file,
                                "data": throughput_mbps
                            })
                except Exception as e:
                    print(f"⚠️ Gagal memproses {file}: {e}")
            
            if self.traces:
                print(f"✅ Berhasil memuat dan mengonversi {len(self.traces)} file trace Mahimahi.")
        
        if not self.traces:
            print("⚠️ Folder trace kosong. Menggunakan pola sintetis Mbps...")
            self.traces.append({
                "name": "synth_volatile",
                "data": np.clip(10 + 5 * np.sin(np.linspace(0, 50, 1000)) + np.random.normal(0, 2, 1000), 0.5, 20).tolist()
            })

        self.active_trace = None
        self.ptr = 0

    def select_random_trace(self):
        """Memilih trace acak untuk episode baru agar agen belajar generalisasi pola."""
        self.active_trace = random.choice(self.traces)
        if len(self.active_trace["data"]) > 105:
            self.ptr = random.randint(0, len(self.active_trace["data"]) - 105)
        else:
            self.ptr = 0
        return self.active_trace["name"]

    def get_next_bandwidth(self):
        if not self.active_trace:
            self.select_random_trace()
        
        val = self.active_trace["data"][self.ptr]
        self.ptr = (self.ptr + 1) % len(self.active_trace["data"])
        return val

class TandonMahimahiEnv(gym.Env):
    """
    World Agent berbasis Multi-Trace Mahimahi dengan Logika Operator Tandon Air.
    """
    def __init__(self, trace_manager):
        super(TandonMahimahiEnv, self).__init__()
        self.trace_manager = trace_manager
        self.bitrates = [0.5, 2.5, 8.0] # Mbps (Low, Mid, High)
        
        # State Space: [Buffer, Throughput, LastAction, Buffer_Trend, TP_Trend, RTT, Dropped]
        self.observation_space = spaces.Box(
            low=np.array([0, 0, 0, -10, -20, 0, 0]),
            high=np.array([30, 50, 2, 10, 20, 1000, 100]),
            dtype=np.float32
        )
        self.action_space = spaces.Discrete(3)
        
        self.state = None
        self.max_steps = 100
        self.current_step = 0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        trace_name = self.trace_manager.select_random_trace()
        initial_tp = self.trace_manager.get_next_bandwidth()
        
        self.state = np.array([15.0, initial_tp, 1.0, 0.0, 0.0, 40.0, 0.0], dtype=np.float32)
        self.current_step = 0
        return self.state, {"trace": trace_name}

    def step(self, action):
        buffer, last_tp_avg, last_action, _, _, rtt, dropped = self.state
        chosen_bitrate = self.bitrates[action]
        
        raw_tp = self.trace_manager.get_next_bandwidth()
        
        seg_duration = 5.0
        download_time = (chosen_bitrate * seg_duration / (raw_tp + 0.1)) + (rtt / 1000.0)
        stalling = max(0, download_time - buffer)
        
        new_buffer = max(0, buffer - download_time) + seg_duration
        new_buffer = min(new_buffer, 30.0)
        
        buf_trend = np.clip(new_buffer - buffer, -10, 10)
        tp_trend = np.clip(raw_tp - last_tp_avg, -20, 20)
        
        # --- LOGIKA REWARD (Safety First) ---
        reward = chosen_bitrate * 1.5 
        reward -= abs(action - last_action) * 1.0 
        
        if stalling > 0:
            reward -= 200.0 # Penalti fatal jika tandon kering
        
        if new_buffer < 10.0:
            reward -= 30.0 
            if action == 2: 
                reward -= 50.0 # Penalti karena nekat di zona bahaya
            
        if action == 0 and raw_tp > 10.0 and new_buffer > 20.0:
            reward -= 15.0 # Penalti terlalu pelit

        self.state = np.array([new_buffer, raw_tp, float(action), buf_trend, tp_trend, 40.0, 0.0], dtype=np.float32)
        self.current_step += 1
        done = self.current_step >= self.max_steps
        
        return self.state, reward, done, False, {"raw_tp": raw_tp}

def save_plots(log_folder, eval_histories):
    """
    Menyimpan semua grafik hasil pelatihan ke folder log.
    """
    # 1. Plot Learning Curve
    try:
        x, y = ts2xy(load_results(log_folder), 'timesteps')
        y_df = pd.Series(y)
        y_smooth = y_df.rolling(window=50, min_periods=1).mean()

        plt.figure(figsize=(10, 5))
        plt.plot(x, y, alpha=0.3, color='gray', label='Reward per Episode')
        plt.plot(x, y_smooth, color='blue', linewidth=2, label='Rata-rata Reward (Smoothed)')
        plt.xlabel('Timesteps')
        plt.ylabel('Skor Reward')
        plt.title("Track Record Pembelajaran Agen RL")
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        plot_path = os.path.join(log_folder, "learning_curve.png")
        plt.savefig(plot_path)
        plt.close()
        print(f"📈 Grafik Learning Curve disimpan di: {plot_path}")
    except Exception as e:
        print(f"⚠️ Gagal menyimpan Learning Curve: {e}")

    # 2. Plot Evaluasi Per Trace
    for i, (name, df) in enumerate(eval_histories):
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
        
        ax1.plot(df.index, df['Throughput'], label=f'Network ({name})', color='blue', alpha=0.5)
        ax1.step(df.index, df['Action']*4 + 2, label='AI Decision', color='red', where='post')
        ax1.set_title(f"Evaluasi Agen pada Trace: {name}")
        ax1.set_ylabel("Mbps / Kualitas")
        ax1.legend()
        ax1.grid(True, alpha=0.2)
        
        ax2.fill_between(df.index, df['Buffer'], color='green', alpha=0.2, label='Level Buffer')
        ax2.axhline(y=10, color='orange', linestyle='--', label='Emergency (10s)')
        ax2.set_ylabel("Detik")
        ax2.set_xlabel("Segmen")
        ax2.legend()
        ax2.grid(True, alpha=0.2)
        
        eval_path = os.path.join(log_folder, f"eval_result_{i}_{name}.png")
        plt.tight_layout()
        plt.savefig(eval_path)
        plt.close()
        print(f"📊 Grafik Evaluasi {name} disimpan di: {eval_path}")

def run_multi_trace_training():
    print("🎬 Memulai Pelatihan RL dengan Konversi Mahimahi Asli...")
    
    log_dir = "./rl_logs/"
    os.makedirs(log_dir, exist_ok=True)
    
    tm = MahimahiTraceManager(folder_path="traces/mahimahi") 
    env = TandonMahimahiEnv(tm)
    env = Monitor(env, log_dir)
    
    model = PPO("MlpPolicy", env, verbose=1, 
                learning_rate=0.0003, 
                n_steps=2048, 
                batch_size=128,
                ent_coef=0.02)
    
    model.learn(total_timesteps=200000)
    model.save("ndn_video_brain_tandon_multi_trace")
    print("✅ Model '.zip' telah diperbarui.")

    # Melakukan Evaluasi pada 3 Trace acak untuk disimpan grafiknya
    eval_histories = []
    print("\n--- Menjalankan Evaluasi Pasca-Latih ---")
    for _ in range(3):
        obs, info = env.reset()
        name = info["trace"]
        history = []
        for _ in range(100):
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, done, _, info_step = env.step(action)
            history.append({
                'Throughput': info_step['raw_tp'],
                'Buffer': obs[0],
                'Action': action
            })
        eval_histories.append((name, pd.DataFrame(history)))

    # Simpan semua plot ke file
    save_plots(log_dir, eval_histories)

if __name__ == "__main__":
    run_multi_trace_training()

🎬 Memulai Pelatihan RL dengan Konversi Mahimahi Asli...
✅ Berhasil memuat dan mengonversi 8 file trace Mahimahi.
Using cpu device
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 100      |
|    ep_rew_mean     | -62      |
| time/              |          |
|    fps             | 5219     |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 100         |
|    ep_rew_mean          | -238        |
| time/                   |             |
|    fps                  | 4257        |
|    iterations           | 2           |
|    time_elapsed         | 0           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.014931556 |
|    clip_fraction        | 0.352